# Notebook 1: Cleaner & Normalizer Test

**Objective**: Scrutinize header detection, cleaning, and normalization logic.

**Key questions**:
- Does auto header detection work for this file?
- Which columns are flagged as multi-valued?
- What bridge tables does normalization create?

In [ ]:


# def find_file(target_name):
#     """Searches likely paths for the test file."""
#     candidates = [
#         project_root / "data" / "raw" / target_name,
#         project_root / "data" / "synthetic" / target_name,
#         project_root / target_name,
#         current_dir / target_name
#     ]
#     for p in candidates:
#         if p.exists():
#             return p
#     return None

# # 4. Locate File
# TARGET_FILE = "professional_incident_tickets.csv"#"SLP_SEA_2025_Core.xlsx"
# file_path = find_file(TARGET_FILE)

# if file_path:
#     log(f"Found file at: {file_path}", "ok")
#     log(f"Project Root detected as: {project_root}")
# else:
#     log(f"File '{TARGET_FILE}' not found!", "err")
#     log(f"Please place it in: {project_root / 'data' / 'raw'}", "warn")
    
#     # Fallback: Search for ANY xlsx file in data/raw to help you debug
#     alt_files = list((project_root / "data" / "raw").glob("*.xlsx"))
#     if alt_files:
#         log(f"I found these other Excel files there:", "info")
#         for f in alt_files:
#             display(Markdown(f"- `{f.name}`"))
#     else:
#         log("data/raw/ directory is empty or missing.", "warn")

# # Set global variable for the rest of the notebook
# TEST_FILE_PATH = str(file_path) if file_path else None

In [1]:
# ── Setup & File Discovery ─────────────────────────────────────────────────────
import sys, os, glob, json
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown, HTML

# 1. Smart Path Resolution
# Notebooks usually run from their own folder (e.g. hybridtablerag/notebooks/)
current_dir = Path.cwd()
if current_dir.name == "notebooks":
    project_root = current_dir.parent
else:
    project_root = current_dir

# Ensure project root is on sys.path
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# 2. Imports
from hybridtablerag.core.cleaner import read_file, clean_dataframe
from hybridtablerag.core.profiler import profile_dataframe
from hybridtablerag.core.normalizer import Normalizer
from hybridtablerag.llm.factory import get_llm

# 3. Helpers
def log(msg, level="info"):
    colors = {"info": "#2563eb", "warn": "#b45309", "err": "#dc2626", "ok": "#16a34a"}
    display(HTML(f"<div style='font-family:monospace;font-size:.85rem;color:{colors.get(level,'#0f172a')}'>› {msg}</div>"))


In [2]:

#  Load File Using cleaner.py 

#FILE_PATH = r"c:\Users\Nikita.Dey\Documents\training py\HybridTableRag\data\raw\SLP_SEA_2025_Core.xlsx"
FILE_PATH = r"c:\Users\Nikita.Dey\Documents\training py\HybridTableRag\data\synthetic\professional_incident_tickets.csv"


# ✅ For two-row header: pass [0, 1]
sheets = read_file(FILE_PATH, header_rows=[0, 1])

print(f"Loaded sheets: {list(sheets.keys())}")

# ── Step 2: Clean (cell-level only) ──────────────────────────
cleaned_sheets = {}
for name, df in sheets.items():
    cleaned, log = clean_dataframe(df, log=[])
    cleaned_sheets[name] = cleaned
    
    display(Markdown(f"### Cleaned: `{name}`"))
    display(Markdown(f"**Columns**: `{list(cleaned.columns)}`"))
    display(cleaned.head(3))
    

Loaded sheets: ['sheet']


### Cleaned: `sheet`

**Columns**: `['ticket_ticket_id', 'ticket_created_date', 'ticket_resolved_date', 'ticket_priority', 'ticket_impacted_departments', 'customer_name', 'customer_email', 'resolution_affected_systems', 'resolution_actions_taken', 'resolution_description']`

,ticket_ticket_id,ticket_created_date,ticket_resolved_date,ticket_priority,ticket_impacted_departments,customer_name,customer_email,resolution_affected_systems,resolution_actions_taken,resolution_description
0,TCKT-1000,2024-05-03,2024-05-29,Medium,Finance;IT;Operations,Gabriella Davis,trevinomichele@example.net,"[{""system"": ""Payroll System"", ""impact"": ""Full""}]","[{""step"": ""Involve gun democratic several incl...",Of woman energy trade suddenly process you. Pe...
1,TCKT-1001,2025-03-31,2025-04-28,Low,HR;IT,Tiffany Howell,NaN,"[{""system"": ""Payroll System"", ""impact"": ""Parti...","[{""step"": ""True car site."", ""performed_by"": ""C...",Machine bar minute. Traditional us production.
2,TCKT-1002,2024-07-10,2024-08-01,Medium,Sales;Operations,Brandon Maxwell,joescott@example.com,"[{""system"": ""Database"", ""impact"": ""Full""}, {""s...","[{""step"": ""Read forget issue Democrat kitchen ...",Brother build scene interview. Later idea assu...


In [ ]:
# ── Manual Override Test: Force header_rows ───────────────────────────
# Uncomment to test manual header specification

# MANUAL_HEADER_ROWS = [0]  # or [0,1] for multi-level
# log(f"Re-reading with manual header_rows={MANUAL_HEADER_ROWS}")
# sheets_manual = read_file(FILE_PATH, header_rows=MANUAL_HEADER_ROWS)
# for name, df in sheets_manual.items():
#     display(Markdown(f"### Manual override: `{name}`"))
#     display(df.head(2))

In [ ]:
# Profiling Test

profiles = {}
for name, df in cleaned_sheets.items():
    profile = profile_dataframe(df)
    profiles[name] = profile
    
    display(Markdown(f"### Profile: `{name}`"))
    
    # Build summary table
    summary_rows = []
    for col, stats in profile["columns"].items():
        summary_rows.append({
            "column": col,
            "dtype": stats["dtype"],
            "nulls": stats["num_nulls"],
            "% null": f"{stats['pct_null']:.1f}",
            "unique": stats["num_unique"],
            "multi?": True if stats.get("is_multi_valued") else "",
            "sample": " | ".join(str(v) for v in stats["sample_values"][:2]),
        })
    display(pd.DataFrame(summary_rows).set_index("column"))
    
    # Highlight multi-valued columns
    multi_cols = [c for c,s in profile["columns"].items() if s.get("is_multi_valued")]
    if multi_cols:
        print(f"Multi-valued columns (will become bridge tables): {multi_cols}", "warn")
    else:
        print("No multi-valued columns detected", "ok")

### Profile: `sheet`

,dtype,nulls,% null,unique,multi?,sample
column,,,,,,
ticket_ticket_id,str,0,0.0,1000,,TCKT-1000 | TCKT-1001
ticket_created_date,str,0,0.0,556,,2024-05-03 | 2025-03-31
ticket_resolved_date,str,0,0.0,570,,2024-05-29 | 2025-04-28
ticket_priority,str,0,0.0,4,,Medium | Low
ticket_impacted_departments,str,0,0.0,212,,Finance;IT;Operations | HR;IT
customer_name,str,0,0.0,984,,Gabriella Davis | Tiffany Howell
customer_email,str,48,4.8,950,,trevinomichele@example.net | joescott@example.com
resolution_affected_systems,str,0,0.0,433,,"[{""system"": ""Payroll System"", ""impact"": ""Full""..."
resolution_actions_taken,str,0,0.0,1000,,"[{""step"": ""Involve gun democratic several incl..."


No multi-valued columns detected ok


In [7]:
# ── Normalization Test ────────────────────────────────────────────────
from hybridtablerag.core.normalizer import NormalizationPlan


print("Testing Normalizer with LLM…")

llm = get_llm()  # Reads from .env
normalizer = Normalizer(llm=llm)

norm_results = {}
for name, df in cleaned_sheets.items():
    print(f"Normalizing `{name}`…")
    
    try:
        plan: NormalizationPlan = normalizer.normalize(
            df, 
            table_hint=name.replace(" ", "_").lower(),
            log=[]
        )
        
        norm_results[name] = plan
        
        # Show results
        display(Markdown(f"#### Normalization Plan: `{name}`"))
        display(Markdown(f"**Main table**: `{plan.main_table_name}` ({len(plan.main_df)} rows)"))
        
        if plan.bridge_tables:
            display(Markdown("**Bridge tables created**:"))
            for bt in plan.bridge_tables:
                display(Markdown(
                    f"- `{bt.name}`: {len(bt.df)} rows | "
                    f"cols: {list(bt.df.columns)} | "
                    f"from: `{bt.source_col}` (sep: `{bt.separator}`)"
                ))
        else:
            display(Markdown("No bridge tables needed"))
        
        if plan.relationships:
            display(Markdown("**Relationships**:"))
            for r in plan.relationships:
                display(Markdown(
                    f"- `{r['from_table']}.{r['from_column']}` → "
                    f"`{r['to_table']}.{r['to_column']}` ({r['type']})"
                ))
                
    except Exception as e:
        print(f"Normalization failed for `{name}`: {e}", "err")
        import traceback
        display(Markdown(f"```\n{traceback.format_exc()}\n```"))

Testing Normalizer with LLM…
Normalizing `sheet`…


#### Normalization Plan: `sheet`

**Main table**: `sheet` (1000 rows)

**Bridge tables created**:

- `sheet_ticket_impacted_department`: 1996 rows | cols: ['ticket_ticket_id', 'ticket_impacted_department'] | from: `ticket_impacted_departments` (sep: `;`)

...

- `sheet_resolution_actions_taken`: 2032 rows | cols: ['ticket_ticket_id', 'step', 'performed_by'] | from: `resolution_actions_taken` (sep: `json`)

**Relationships**:

- `sheet_ticket_impacted_department.ticket_ticket_id` → `sheet.ticket_ticket_id` (many_to_one)

- `sheet_resolution_affected_system.ticket_ticket_id` → `sheet.ticket_ticket_id` (many_to_one)

- `sheet_resolution_actions_taken.ticket_ticket_id` → `sheet.ticket_ticket_id` (many_to_one)

In [8]:
# ── Summary & Export ──────────────────────────────────────────────────
display(Markdown("## Test Complete"))

summary = {
    "file": FILE_PATH,
    "sheets_processed": list(sheets.keys()),
    "normalization_results": {
        name: {
            "main_table": plan.main_table_name,
            "bridge_count": len(plan.bridge_tables),
            "relationships": plan.relationships,
        }
        for name, plan in norm_results.items()
    }
}

display(Markdown("```json\n" + json.dumps(summary, indent=2) + "\n```"))


## Test Complete

```json
{
  "file": "c:\\Users\\Nikita.Dey\\Documents\\training py\\HybridTableRag\\data\\synthetic\\professional_incident_tickets.csv",
  "sheets_processed": [
    "sheet"
  ],
  "normalization_results": {
    "sheet": {
      "main_table": "sheet",
      "bridge_count": 3,
      "relationships": [
        {
          "from_table": "sheet_ticket_impacted_department",
          "from_column": "ticket_ticket_id",
          "to_table": "sheet",
          "to_column": "ticket_ticket_id",
          "type": "many_to_one"
        },
        {
          "from_table": "sheet_resolution_affected_system",
          "from_column": "ticket_ticket_id",
          "to_table": "sheet",
          "to_column": "ticket_ticket_id",
          "type": "many_to_one"
        },
        {
          "from_table": "sheet_resolution_actions_taken",
          "from_column": "ticket_ticket_id",
          "to_table": "sheet",
          "to_column": "ticket_ticket_id",
          "type": "many_to_one"
        }
      ]
    }
  }
}
```